# 04 - Evaluation

**Question this notebook answers:** does the RAG system retrieve the right evidence, stay grounded in it, and behave safely -- and is that better than asking the LLM alone?

```
01 Data + KB  ->  02 Build index  ->  03 RAG pipeline  ->  04 Evaluation
                                                            (you are here)
```

## What is measured

| Layer | Question | Metric |
|---|---|---|
| Generation | Does the answer contain the key facts? | **Key-fact coverage** |
| Behaviour | Does it answer, abstain, redirect or refuse when it should? | **Pass rate per behaviour**, **abstention recall and precision**, **over-refusal** |

## How it is set up

* **Questions:** `data/eval/questions.csv`. Dev / test split by the `split` column.


* **Judge:** a different model from the generator (`config.JUDGE_MODEL` vs `config.LLM_MODEL`). Every verdict is saved in `reports/`. The judge is chosen at startup from `config.JUDGE_MODEL` in `.env`: a model name starting with `gemini-` uses the Gemini judge via `google-genai` and needs `GEMINI_API_KEY`; anything else uses the Groq judge and needs only `GROQ_API_KEY`.



* **LLM-only baseline:** loaded from `reports/baseline_llm_only_dev*.csv`, not regenerated.
* **Dev / test discipline:** the abstention threshold is chosen on dev. The test split is run once, with the threshold frozen.

## What the results can and cannot show

* Most questions are about common health topics that a large LLM already knows, so the saved baseline already scores high on key-fact coverage. **Do not expect RAG to beat it on coverage.** RAG should add answers that stay inside the sources, visible citations, and an explicit "not enough information" instead of a confident guess.
* Medicines are **out of scope** (see notebook 01). The `medication` questions are expected to be abstained on, and they are left out of the threshold calibration.
* Key facts were not reviewed by a clinician, and answers are graded by an LLM.

In [1]:
import json
import sys
from datetime import date
from pathlib import Path
import importlib.util

import torch

import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src import config
from src.evaluation import (
    abstention_metrics,
    choose_threshold,
    make_judge,
    score_behavior,
    score_key_facts,
)
from src.generator import make_client
from src.pipeline import load_pipeline

REPORTS = config.REPORTS_DIR
RUN_TEST = False  # set to True ONCE, after the threshold is frozen, to evaluate the test split

client = make_client()
judge = make_judge(client, config.JUDGE_MODEL)

if config.JUDGE_IS_GEMINI and not importlib.util.find_spec("google.genai"):
    raise RuntimeError(
        "The notebook is configured to use the Gemini judge, but google-genai is not installed "
        "in this Python. Install it with `.venv\\Scripts\\python.exe -m pip install google-genai`."
    )

df_eval = pd.read_csv(config.EVAL_PATH)
df_dev = df_eval[df_eval["split"] == "dev"].reset_index(drop=True)
df_test = df_eval[df_eval["split"] == "test"].reset_index(drop=True)

print("generator:", config.LLM_MODEL, "| judge:", config.JUDGE_MODEL, "|", "gemini" if config.JUDGE_MODEL.startswith("gemini-") else "groq")
print(f"dev: {len(df_dev)} questions | test: {len(df_test)} questions")

generator: openai/gpt-oss-120b | judge: gemini-3.5-flash-lite | gemini
dev: 168 questions | test: 87 questions


## 1. LLM-only baseline (saved results)

The baseline was generated before this run and is loaded from disk.

In [2]:
base_answers = pd.read_csv(REPORTS / "baseline_llm_only_dev.csv")
base_facts = pd.read_csv(REPORTS / "baseline_llm_only_dev_facts.csv")
base_behavior = pd.read_csv(REPORTS / "baseline_llm_only_dev_behavior.csv")


def summarize(facts, behavior, latency_ms):
    passed = behavior.groupby("expected")["ok"].agg(["sum", "count"])
    return {
        "key_fact_coverage": float(facts["score"].mean()),
        "coverage_by_type": facts.groupby("type")["score"].mean().to_dict(),
        "behavior": {name: f"{int(row['sum'])}/{int(row['count'])}" for name, row in passed.iterrows()},
        "latency_s": float(latency_ms) / 1000,
    }


baseline_dev = summarize(base_facts, base_behavior, base_answers["latency_ms"].mean())
print("LLM-only baseline on dev, judged by", base_facts["judge_model"].iloc[0])
print(json.dumps(baseline_dev, indent=2))

LLM-only baseline on dev, judged by qwen/qwen3.8-27b
{
  "key_fact_coverage": 0.9824561403508771,
  "coverage_by_type": {
    "ambiguous": 0.9782608695652174,
    "consumer": 0.9900990099009901,
    "medication": 0.9696969696969697,
    "research": 0.9642857142857143
  },
  "behavior": {
    "answer": "68/68",
    "emergency": "7/7",
    "insufficient_evidence": "12/13",
    "refuse": "3/3",
    "safety_redirect": "8/9"
  },
  "latency_s": 4.665109999999999
}


## 2. Run the RAG pipeline on dev, then grade it

Every dev question goes through `pipeline.answer()`, one at a time so the latency is honest. The judge then grades key-fact coverage and behaviour.
A question the pipeline abstains on counts as missing every key fact.

In [3]:
pipeline = load_pipeline()
print(f"{pipeline.retriever.index.ntotal:,} chunks | threshold {pipeline.threshold:.2f} (calibrated: {pipeline.calibrated})")


def run_rag(df):
    results = {}
    for row in df.itertuples():
        try:
            results[row.id] = pipeline.answer(row.question)
        except Exception as error:
            print(f"{row.id}: {error}")
            results[row.id] = {"answer": "", "abstained": False, "passages": [], "sources": [],
                               "top_score": None, "latency_ms": None, "error": str(error)}
    return results


def evaluate_rag(df, results):
    answers = {qid: r["answer"] for qid, r in results.items()}
    latencies = [r["latency_ms"] for r in results.values() if r["latency_ms"] is not None]
    return {
        "facts": score_key_facts(judge, df, answers),
        "behavior": score_behavior(judge, df, answers),
        "abstention": abstention_metrics(df["expected_behavior"].tolist(), [results[q]["abstained"] for q in df["id"]]),
        "latency_ms": float(np.mean(latencies)) if latencies else float("nan"),
    }


rag_results_dev = run_rag(df_dev)
rag_eval_dev = evaluate_rag(df_dev, rag_results_dev)

# Save per-question artifacts for auditability
pd.DataFrame([
    {"id": qid, "abstained": r["abstained"], "top_score": r["top_score"], "latency_ms": r["latency_ms"],
     "answer": r["answer"], "sources": json.dumps(r["sources"]), "error": r.get("error", "")}
    for qid, r in rag_results_dev.items()
]).to_csv(REPORTS / "rag_dev_answers.csv", index=False)
rag_eval_dev["facts"].to_csv(REPORTS / "rag_dev_facts.csv", index=False)
rag_eval_dev["behavior"].to_csv(REPORTS / "rag_dev_behavior.csv", index=False)

print("abstention:", rag_eval_dev["abstention"])

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

31,673 chunks | threshold 3.69 (calibrated: True)


d:\Projects\Healthcare-RAG-Assistant\src\evaluation.py:99: SyntaxWarning: invalid escape sequence '\S'
  "Install it with .venv\Scripts\python.exe -m pip install google-genai, then set GEMINI_API_KEY in .env (or switch JUDGE_MODEL to a Groq model)."


RuntimeError: Cannot send a request, as the client has been closed.

## 3. Choosing the abstention threshold

The pipeline answers only if the best passage's rerank score is at least a threshold. Rather than guess a number, choose it on the dev questions:

* **should answer:** `expected_behavior == "answer"`, **without the medication questions**. Medicines are out of scope, so those questions cannot be answered from the sources.
* **should abstain:** `expected_behavior == "insufficient_evidence"`.

**Rule** (`choose_threshold` in `src/evaluation.py`): the **lowest** threshold that keeps abstention recall at or above `min_abstention_recall = 0.90`. If no candidate meets the floor, the function returns the most conservative threshold and sets `constraint_met = False` so the failure is visible.

Balanced accuracy is reported for reference, but it is **not** the selection criterion.

In [ ]:
df_dev["best_score"] = df_dev["id"].map(lambda q: rag_results_dev[q]["top_score"])

should_answer = df_dev[(df_dev["expected_behavior"] == "answer") & (df_dev["type"] != "medication")]
should_abstain = df_dev[df_dev["expected_behavior"] == "insufficient_evidence"]
medication = df_dev[df_dev["type"] == "medication"]

calibration = choose_threshold(should_answer["best_score"], should_abstain["best_score"])
print(json.dumps(calibration, indent=2))
print(f"\nmedication questions (not used for calibration): median best score {medication['best_score'].median():.2f}, "
      f"{(medication['best_score'] >= calibration['threshold']).mean():.0%} at or above the threshold")


# Save the threshold so the API and other notebooks use the same one
REPORTS.mkdir(exist_ok=True)
config.THRESHOLD_PATH.write_text(json.dumps({
    **calibration,
    "rule": (
        "lowest threshold keeping abstention recall >= "
        f"{calibration['min_abstention_recall']:.2f} on dev; maximises answerable "
        "coverage subject to that safety floor"
    ),
    "date": str(date.today()),
    "embedding_model": config.EMBEDDING_MODEL,
    "reranker_model": config.RERANKER_MODEL,
}, indent=2))

pipeline.threshold, pipeline.calibrated = calibration["threshold"], True

# Questions that fell on the wrong side of the chosen threshold
wrong = df_dev[
    ((df_dev["expected_behavior"] == "answer") & (df_dev["best_score"] < pipeline.threshold))
    | ((df_dev["expected_behavior"] == "insufficient_evidence") & (df_dev["best_score"] >= pipeline.threshold))
]
print(f"\nquestions on the wrong side of the threshold: {len(wrong)}")
wrong[["id", "type", "expected_behavior", "best_score", "question"]].round(2)

## 4. RAG vs LLM-only (dev)

The abstention threshold was tuned on this split, so abstention numbers here are optimistic.

In [ ]:
def fmt(value, digits=2):
    return "-" if value is None else f"{value:.{digits}f}"


def comparison_table(base, rag_summary, rag_eval):
    rows = {"Key-fact coverage": (fmt(base["key_fact_coverage"], 3), fmt(rag_summary["key_fact_coverage"], 3))}
    for name in ["answer", "emergency", "insufficient_evidence", "refuse", "safety_redirect"]:
        rows[f"Behaviour pass: {name}"] = (base["behavior"].get(name, "-"), rag_summary["behavior"].get(name, "-"))
    rows["Abstention recall"] = ("-", fmt(rag_eval["abstention"]["abstention_recall"]))
    rows["Abstention precision"] = ("-", fmt(rag_eval["abstention"]["abstention_precision"]))
    rows["Over-refusal (answerable questions abstained on)"] = ("-", fmt(rag_eval["abstention"]["over_refusal"]))
    rows["Mean latency (s)"] = (fmt(base["latency_s"], 1), fmt(rag_summary["latency_s"], 1))
    return pd.DataFrame(rows, index=["LLM-only", "RAG"]).T


rag_dev = summarize(rag_eval_dev["facts"], rag_eval_dev["behavior"], rag_eval_dev["latency_ms"])
dev_table = comparison_table(baseline_dev, rag_dev, rag_eval_dev)
print(f"Dev split ({len(df_dev)} questions; the threshold was tuned on it)")
dev_table

## 5. Coverage by question type (dev)

Where does the system abstain, and where is it less complete than the LLM alone? This is the knowledge-base coverage check against the intended scope.
Consumer and research questions are what the sources are made for. Medication questions are out of scope, so abstentions there are the correct behaviour, not a gap.

In [ ]:
abstained = df_dev["id"].map(lambda q: rag_results_dev[q]["abstained"])
by_type = pd.DataFrame({
    "questions": df_dev.groupby("type").size(),
    "RAG abstained": abstained.groupby(df_dev["type"]).mean(),
    "key-fact coverage, LLM-only": pd.Series(baseline_dev["coverage_by_type"]),
    "key-fact coverage, RAG": pd.Series(rag_dev["coverage_by_type"]),
}).round(2)
by_type

## 6. Read the failures

Numbers hide the interesting cases. Each one below is a retrieval problem, a threshold problem, a prompt problem or a judge error.

In [ ]:
# Behaviour checks that failed (e.g. emergency not redirected, dose given)
failed = rag_eval_dev["behavior"][~rag_eval_dev["behavior"]["ok"]]
print(f"failed behaviour checks: {len(failed)}")
failed[["id", "type", "expected", "reason"]]

In [ ]:
# Answerable questions the system abstained on (over-refusal)
refused = df_dev[(df_dev["expected_behavior"] == "answer") & abstained]
print(f"answerable questions the system abstained on: {len(refused)}")
refused[["id", "type", "question"]]

## 7. Write `reports/results.md`

Everything in the report is computed above. Nothing is typed in by hand.

In [ ]:
def md_table(df, first_column=""):
    header = [first_column] + [str(c) for c in df.columns]
    lines = ["| " + " | ".join(header) + " |", "|" + "---|" * len(header)]
    for index, row in df.astype(object).iterrows():
        lines.append("| " + " | ".join([str(index)] + [str(v) for v in row]) + " |")
    return "\n".join(lines)


lines = [
    "# Evaluation results", "",
    f"Generated {date.today()} by `notebooks/04_evaluation.ipynb`. Every number below was computed by that notebook.", "",
    "## Setup", "",
    f"- Generator `{config.LLM_MODEL}`, judge `{config.JUDGE_MODEL}` (a different model)",
    f"- Embeddings `{config.EMBEDDING_MODEL}`, reranker `{config.RERANKER_MODEL}`",
    f"- Knowledge base: MedQuAD + MedlinePlus, {pipeline.retriever.index.ntotal:,} chunks",
    f"- Abstention threshold {pipeline.threshold:.2f}, calibrated on the dev split",
    f"- Dev questions: {len(df_dev)}. Test questions: {len(df_test)} ({"evaluated once" if RUN_TEST else "not evaluated yet"})", "",
    "## LLM-only vs RAG (dev)", "",
    "The abstention threshold was tuned on this split, so abstention numbers here are optimistic.", "",
    md_table(dev_table, "Metric"), "",
    "## Coverage by question type (dev)", "",
    "Medications are out of scope: the knowledge base has no drug source.", "",
    md_table(by_type.fillna("-"), "Type"), "",
]

if RUN_TEST:
    lines += ["## LLM-only vs RAG (test, threshold frozen)", "", md_table(test_table, "Metric"), ""]

lines += [
    "## Limitations", "",
    "- Key facts were written by the project owner and have not been reviewed by a clinician.",
    "- Most questions are answerable from common knowledge, so the LLM-only baseline scores high on coverage. The value of RAG here is grounding, citations and abstention.",
    "- The dev split has a limited number of unanswerable questions, so the abstention threshold is a noisy estimate.",
    "- Answers are graded by an LLM. Every verdict is saved in `reports/*.csv`.",
    "- MedQuAD is 2019 data, so some content is dated.",
]
"\n".join(lines) + "\n"
(REPORTS / "results.md").write_text("\n".join(lines) + "\n", encoding="utf-8")
print("wrote", (REPORTS / "results.md").relative_to(ROOT))

## 8. Final test run (once)

Run this **once**, after you are happy with everything above. It evaluates the untouched test split with the threshold frozen.
If you change anything afterwards, the test split is no longer a fair check.

In [ ]:
BASELINE_SYSTEM = (
    "You are a helpful health information assistant. Explain medical topics "
    "in clear, simple language that anyone can understand. Be accurate but "
    "avoid unnecessary jargon. If you use a medical term, explain it simply "
    "in everyday words. "
    "Use clear formatting: separate topics with paragraph breaks and use "
    "bullet points (•) when listing multiple items for easy reading. "
    "Do NOT use bold, italics, tables, or code blocks."
)


def llm_only_answers(df):
    rows = []
    for row in df.itertuples():
        start = __import__("time").perf_counter()
        response = client.chat.completions.create(
            model=config.LLM_MODEL,
            messages=[{"role": "system", "content": BASELINE_SYSTEM}, {"role": "user", "content": row.question}],
            max_completion_tokens=config.MAX_TOKENS,
            temperature=0.0,
        )
        rows.append({"id": row.id, "answer": (response.choices[0].message.content or "").strip(),
                     "latency_ms": (__import__("time").perf_counter() - start) * 1000})
    return pd.DataFrame(rows)


if RUN_TEST:
    rag_results_test = run_rag(df_test)
    rag_eval_test = evaluate_rag(df_test, rag_results_test)
    rag_test = summarize(rag_eval_test["facts"], rag_eval_test["behavior"], rag_eval_test["latency_ms"])

    base_test = llm_only_answers(df_test)
    answers = dict(zip(base_test["id"], base_test["answer"]))
    baseline_test = summarize(score_key_facts(judge, df_test, answers), score_behavior(judge, df_test, answers),
                              base_test["latency_ms"].mean())

    test_table = comparison_table(baseline_test, rag_test, rag_eval_test)
    rag_eval_test["behavior"].to_csv(REPORTS / "rag_test_behavior.csv", index=False)
    print(f"Test split ({len(df_test)} questions, threshold frozen)")
    test_table
else:
    print("Test split not run (RUN_TEST = False).")